## **Lap & Weather Integration**

##### **Imports**

In [ ]:
import pandas as pd

from pathlib import Path

In [ ]:
# Import and combine all f1 lap data
weather_directory = Path('../data/ff1_weather_data')
weather_files = weather_directory.glob('weather_data_*.csv')

weather_data = pd.concat(
    [pd.read_csv(file) for file in weather_files],
    ignore_index=True
)

lap_data = pd.read_csv('../data/cleaned_lap_data_2018_2025.csv')

weather_data.head()

#### **Weather Cleaning**

In [ ]:
# Standardize 'Time' column type to timedelta
weather_data['Time'] = pd.to_timedelta(weather_data['Time'])

In [ ]:
# Normalize location names 
weather_data['Location'] = weather_data['Location'].replace({
    'Monte Carlo': 'Monaco',
    'Marina Bay': 'Singapore',
    'Miami Gardens': 'Miami',
    'Yas Island': 'Yas Marina'
})

In [ ]:
# Load event data
event_data = pd.read_csv('../data/event_location_data.csv')

# Set 'StartTime' to datetime and sort values
event_data['StartTime'] = pd.to_datetime(event_data['StartTime'])
event_data.sort_values(['Year', 'Location', 'StartTime'])

event_data['RaceSeq'] = event_data.groupby(['Year', 'Location']).cumcount()

In [ ]:
# (Current row time - next row time < 0) = next race
new_race = (
    weather_data
    .groupby(['Year', 'Location'])['Time']
    .diff() < pd.Timedelta(0)
).astype(int)

weather_data['RaceSeq'] = (
    new_race
    .groupby([weather_data['Year'], weather_data['Location']])
    .cumsum()
)

# Add event name to handle repeat locations
weather_data = weather_data.merge(
    event_data[['Year', 'Location', 'RaceSeq', 'EventName', 'StartTime']],
    on=['Year', 'Location', 'RaceSeq'],
    how='left'
)

# Add a UTC lap start time for merging with weather data
weather_data['TimeUTC'] = weather_data['Time'] + weather_data['StartTime']

weather_data = weather_data.drop(['RaceSeq', 'StartTime'], axis=1)


In [ ]:
# Resampling function
def resample_func(weather):
    weather = weather.copy()
    weather.set_index('TimeUTC', inplace=True) # required for resample
    
    numeric_cols = ["AirTemp", "Humidity", "Pressure", "TrackTemp",
                    "WindDirection", "WindSpeed"]
    
    interpolate_df = weather[numeric_cols].resample("1s").interpolate()
    ffill_df = weather[["Rainfall"]].resample("1s").ffill()
    
    return interpolate_df.join(ffill_df)
    
# Round TimeUTC down to the nearest second
weather_data["TimeUTC"] = weather_data["TimeUTC"].dt.floor("1s")

# Resample weather data
weather_resampled = (
    weather_data
    .groupby(['Year', 'Location', 'EventName'])
    .apply(resample_func, include_groups=False)
    .reset_index()
)

weather_resampled.head()

In [ ]:
# Round TimeUTC down to the nearest second
lap_data["LapStartTimeUTC"] = pd.to_datetime(lap_data["LapStartTimeUTC"]).dt.floor("1s")

# Missing weather data fill to closest available second
lap_data.loc[
    lap_data["LapStartTimeUTC"] == pd.to_datetime('2020-10-11 12:10:07'), 
    'LapStartTimeUTC'
] = pd.to_datetime('2020-10-11 12:10:27')

# Assign weather data per lap
lap_weather_data = pd.merge(
    left=lap_data, 
    right=weather_resampled,
    left_on=['Year', 'Location', 'EventName', 'LapStartTimeUTC'],
    right_on=['Year', 'Location', 'EventName', 'TimeUTC'],
    how='left'
).drop('TimeUTC', axis=1)

lap_weather_data.head()

In [ ]:
# Store combined data
lap_weather_data.to_csv('../data/lap_weather_data_2018_2025.csv', index=False)